# Real integration test: `get_base_model(partial=True)`

Downloads DeepSeek-V2-Lite from HuggingFace, builds a partial model through the real `get_base_model` code path with a real `ExpertManager` loaded from `expert_groups/exp_dummy`, and verifies that backbone + owned-expert tensors match the pretrained checkpoint instead of the random `_init_weights` default.

CPU-only — every weight load and tensor comparison runs in host RAM. VRAM is never touched: the script sets `config.model.device = "cpu"`, so `get_base_model` keeps the partial model on CPU and the pretrained state dict is also loaded on CPU (production miners with a real GPU would land the partial model in VRAM instead, leaving the CPU state dict to drain into VRAM tensor-by-tensor as it streams).

No mocks. Heavy — peak ~40 GB RAM during the streaming load (partial model + the still-being-consumed pretrained state dict), and another ~40 GB peak during the verification compare. First run pulls ~16 GB from HuggingFace; subsequent runs reuse the HF cache.

Run this notebook from the repo root so the `connito` package is importable.

In [1]:
from __future__ import annotations

from pathlib import Path

import torch

from connito.shared.config import MinerConfig
from connito.shared.expert_manager import ExpertManager
from connito.shared.modeling.custom_deepseek_v2_lite import CustomDeekSeekMoE
from connito.shared.modeling.mycelia import (
    get_base_model,
    get_base_tokenizer,
    load_pretrained_state_dict,
)

/home/isabella/crucible/ConnitoAI/.main-venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "expert_groups" / "exp_dummy").is_dir():
    REPO_ROOT = REPO_ROOT.parent

EXP_DUMMY_DIR = REPO_ROOT / "expert_groups" / "exp_dummy"
MODEL_PATH = "deepseek-ai/DeepSeek-V2-Lite"
GROUP_ID = 1

assert EXP_DUMMY_DIR.is_dir(), f"expert_groups/exp_dummy not found at {EXP_DUMMY_DIR}"
print(f"REPO_ROOT     = {REPO_ROOT}")
print(f"EXP_DUMMY_DIR = {EXP_DUMMY_DIR}")

REPO_ROOT     = /home/isabella/crucible/ConnitoAI
EXP_DUMMY_DIR = /home/isabella/crucible/ConnitoAI/expert_groups/exp_dummy


## Helpers

`_assert` prints a PASS/FAIL line and raises on failure. The compare helpers walk the partial model's state dict against the pretrained state dict for backbone keys and against the per-expert assignment map for owned experts.

In [27]:
def _assert(condition: bool, message: str) -> None:
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {message}")
    if not condition:
        raise AssertionError(message)


def _compare_backbone(partial_state: dict, full_state: dict) -> None:
    """Walk every key shared between the partial model and the pretrained HF
    state dict; for shape-matched keys, assert the partial model's tensor
    equals the pretrained tensor."""
    shared = [k for k in partial_state if k in full_state]
    matched = []
    mismatched: list[str] = []
    skipped_shape: list[str] = []

    for key in shared:
        p = partial_state[key]
        f = full_state[key]
        if tuple(p.shape) != tuple(f.shape):
            skipped_shape.append(key)
            continue
        if torch.equal(p.detach().cpu().to(f.dtype), f.detach().cpu()):
            matched.append(key)
        else:
            mismatched.append(key)

    print(f"  shape-matched backbone keys: {len(matched)} / {len(shared) - len(skipped_shape)}")
    if matched:
        print(f"  first 5 matched keys: {matched[:50]}")
    if mismatched:
        print(f"  first 5 mismatched keys: {mismatched[:5]}")
    if skipped_shape:
        print(f"  shape-mismatched keys (expected — sliced experts): {len(skipped_shape)}")

    _assert(
        len(matched) > 0,
        "at least one backbone tensor matches the pretrained checkpoint",
    )
    _assert(
        not mismatched,
        f"every shape-matched key equals the pretrained value (got {len(mismatched)} mismatches)",
    )


def _compare_owned_experts(
    partial_model: CustomDeekSeekMoE,
    full_state: dict,
    expert_group_assignment: dict,
    target_group: int,
) -> None:
    """For each owned (my_idx, org_idx) pair, verify the partial model's
    expert tensor equals the pretrained expert at org_idx."""
    partial_state = partial_model.state_dict()
    assignments = expert_group_assignment.get(target_group, {})

    checked = 0
    mismatched: list[str] = []
    missing_full: list[str] = []

    for layer_idx, layer_pairs in assignments.items():
        if not layer_pairs:
            continue
        for my_idx, org_idx in layer_pairs:
            print(f"  Checking owned expert {my_idx} (pretrained index {org_idx})")
            for proj in ("gate_proj", "up_proj", "down_proj"):
                key = f"model.layers.{layer_idx}.mlp.experts.{my_idx}.{proj}.weight"
                if key not in partial_state:
                    print(f"  Skipping {key} (not in partial state)")
                    continue
                full_key_per_expert = (
                    f"model.layers.{layer_idx}.mlp.experts.{org_idx}.{proj}.weight"
                )
                if full_key_per_expert not in full_state:
                    missing_full.append(full_key_per_expert)
                    print(f"  Skipping {full_key_per_expert} (not in pretrained state)")
                    continue
                p = partial_state[key].detach().cpu()
                f = full_state[full_key_per_expert].detach().cpu()
                if tuple(p.shape) != tuple(f.shape):
                    continue
                if torch.equal(p.to(f.dtype), f):
                    checked += 1
                else:
                    mismatched.append(key)

    print(f"  owned-expert tensors verified: {checked} {len(mismatched)} mismatches")
    
    if mismatched:
        print(f"  first 5 owned-expert mismatches: {mismatched[:5]}")
    
    if missing_full:
        print(f"  pretrained-side keys not found (skipped): {len(missing_full)}")

    _assert(
        checked > 0,
        "at least one owned-expert tensor matches the pretrained slice",
    )
    _assert(
        not mismatched,
        f"every checked owned-expert equals the pretrained slice (got {len(mismatched)} mismatches)",
    )


def _check_not_random(partial_state: dict) -> None:
    """Sanity: HuggingFace `_init_weights` for DeepSeek uses
    `normal_(std=config.initializer_range)` (~0.02). The pretrained embedding
    table has a noticeably different distribution. If the fix regressed and
    weights came back random, every backbone std would sit very close to 0.02."""
    key = "model.embed_tokens.weight"
    _assert(key in partial_state, f"partial state dict contains `{key}`")
    std = float(partial_state[key].detach().float().std().item())
    print(f"  embed_tokens.weight std = {std:.4f}")
    _assert(
        abs(std - 0.02) > 0.005,
        "embed_tokens.weight std differs from the random `_init_weights` default of ~0.02",
    )


def _run_forward_pass(
    partial_model: CustomDeekSeekMoE,
    config: MinerConfig,
) -> None:
    """Run a single CPU forward pass through the partial model.

    Structural smoke test: the partial model only owns the experts for one
    group, and `myceliaSparseMoeBlock` silently skips tokens routed to
    unowned experts (`local_expert_idx == -1`), so the logits are only
    partially informed. We assert the model is runnable end-to-end and
    produces a finite logits tensor of the expected `(batch, seq, vocab)`
    shape — not that the output is semantically meaningful."""
    tokenizer = get_base_tokenizer(config)
    prompt = "Hello, world!"
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"]
    print(f"  prompt        = {prompt!r}")
    print(f"  input_ids     = shape {tuple(input_ids.shape)}, dtype {input_ids.dtype}")

    partial_model.eval()
    with torch.no_grad():
        outputs = partial_model(input_ids=input_ids)

    logits = outputs.logits
    print(f"  logits        = shape {tuple(logits.shape)}, dtype {logits.dtype}")

    expected_shape = (
        input_ids.shape[0],
        input_ids.shape[1],
        partial_model.config.vocab_size,
    )
    _assert(
        tuple(logits.shape) == expected_shape,
        f"logits shape is (batch, seq, vocab) = {expected_shape}",
    )
    _assert(
        bool(torch.isfinite(logits).all().item()),
        "all logits are finite (no NaN / Inf)",
    )

## [1/5] Build `MinerConfig` + `ExpertManager` (real, from disk fixture)

`model.device = "cpu"` keeps any downstream `.to(device=...)` calls off CUDA. `get_base_model` itself never moves the model anywhere, but setting this explicitly makes the CPU-only intent clear.

In [6]:
group_id = GROUP_ID  # change here to test a different group from exp_dummy

print("=" * 72)
print("Real integration test: get_base_model(partial=True)")
print(f"  model_path   = {MODEL_PATH}")
print(f"  group_id     = {group_id}")
print(f"  fixture      = {EXP_DUMMY_DIR}")
print( "  device       = cpu (VRAM untouched)")
print("=" * 72)

config = MinerConfig()
config.role = "miner"
config.model.model_path = MODEL_PATH
config.model.base_arch_model = MODEL_PATH
config.model.device = "cpu"
config.model.precision = "bf16-mixed"
# config.model.use_quantization = False
# config.model.use_unsloth = False
config.model.torch_compile = False
config.task.expert_group_name = "exp_dummy"
config.task.base_path = EXP_DUMMY_DIR.parent
config.task.path = EXP_DUMMY_DIR
config.task.load_all_expert_groups = False
config.task.exp.group_id = group_id
config.task.exp.data.sequence_length = 128

print("\n[1/5] Building MinerConfig + ExpertManager (real, from disk fixture)")
expert_manager = ExpertManager(config)
n_groups = expert_manager.num_expert_groups
print(f"  expert_group_assignment loaded ({n_groups} group(s))")
_assert(
    group_id in expert_manager.expert_group_assignment,
    f"expert_manager.expert_group_assignment contains group_id={group_id}",
)

Real integration test: get_base_model(partial=True)
  model_path   = deepseek-ai/DeepSeek-V2-Lite
  group_id     = 1
  fixture      = /home/isabella/crucible/ConnitoAI/expert_groups/exp_dummy
  device       = cpu (VRAM untouched)
2026-05-12 12:58:14 [info     ] Resolving wallet data from chain coldkey=template_coldkey_name hotkey=template_hotkey_name network=archive
2026-05-12 12:58:18 [warning  ] Cannot find wallet keys; check coldkey/hotkey names or pass --hotkey_name/--coldkey_name coldkey_name=template_coldkey_name error=Generic error: Failed to get hotkey: FileNotFound("Keyfile at: /home/isabella/.bittensor/wallets/template_coldkey_name/hotkeys/template_hotkey_name does not exist.") hotkey_name=template_hotkey_name

[1/5] Building MinerConfig + ExpertManager (real, from disk fixture)
2026-05-12 12:58:18 [info     ] Loading expert assignment for active task only task_path=/home/isabella/crucible/ConnitoAI/expert_groups/exp_dummy
  expert_group_assignment loaded (1 group(s))
  [PASS

## [2/5] Call `get_base_model(partial=True)`

In [7]:
print("\n[2/5] Calling get_base_model(partial=True)")
partial_model = get_base_model(
    config=config,
    expert_manager=expert_manager,
    group_ids=[group_id],
    partial=True,
)
_assert(partial_model is not None, "get_base_model returned a model")
_assert(
    isinstance(partial_model, CustomDeekSeekMoE),
    f"model is CustomDeekSeekMoE (got {type(partial_model).__name__})",
)

partial_state = partial_model.state_dict()
print(f"  partial model state_dict keys: {len(partial_state)}")


[2/5] Calling get_base_model(partial=True)
2026-05-12 12:58:42 [info     ] Initialized MoE expert layout expert_first=8 expert_last=15 full_mode=False layer_id=1 num_local_experts=8 position=first
2026-05-12 12:58:55 [info     ] Initialized MoE expert layout expert_first=16 expert_last=23 full_mode=False layer_id=26 num_local_experts=8 position=last


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 351/351 [00:05<00:00, 62.14it/s]


2026-05-12 12:59:20 [info     ] Streamed pretrained state dict into partial model loaded_counts={'full': 299, 'sliced': 0} target_group=1
2026-05-12 12:59:21 [info     ] Streamed pretrained backbone + owned-expert slices into partial model device=cpu dtype=torch.bfloat16 target_group=1
  [PASS] get_base_model returned a model
  [PASS] model is CustomDeekSeekMoE (got CustomDeekSeekMoE)
  partial model state_dict keys: 715


## [3/5] Sanity-check: weights are not at the random `_init_weights` default

In [8]:
print("\n[3/5] Sanity-check: weights are not at the random `_init_weights` default")
_check_not_random(partial_state)


[3/5] Sanity-check: weights are not at the random `_init_weights` default
  [PASS] partial state dict contains `model.embed_tokens.weight`
  embed_tokens.weight std = 0.1315
  [PASS] embed_tokens.weight std differs from the random `_init_weights` default of ~0.02


## [4/5] Compare partial model tensors against the HF pretrained checkpoint

First run pulls ~16 GB from HuggingFace; subsequent runs reuse the HF cache. We load the state dict in bf16 to match the upstream safetensors storage exactly (no load-time cast). Peak RAM during this cell is ~34 GB.

In [9]:
print("\n[4/5] Comparing partial model tensors against the HF pretrained checkpoint")
print("  Loading pretrained state dict in bf16 on CPU (first run pulls ~16 GB)...")
full_state = load_pretrained_state_dict(MODEL_PATH, dtype=torch.bfloat16)
print(f"  pretrained state dict keys: {len(full_state)}")


[4/5] Comparing partial model tensors against the HF pretrained checkpoint
  Loading pretrained state dict in bf16 on CPU (first run pulls ~16 GB)...


Loading weights: 100%|██████████| 351/351 [00:05<00:00, 61.41it/s]


  pretrained state dict keys: 351


In [34]:
for k in partial_state.keys():
    if "expert" in k:
        print(f"expert key {k}")
        break

for k in full_state.keys():
    if "expert" in k:
        print(f"expert key {k}")
        break

expert key model.layers.1.mlp.experts.8.gate_up_proj
expert key model.layers.1.mlp.experts.gate_up_proj


In [28]:
print("\n  --- Backbone ---")
_compare_backbone(partial_state, full_state)


  --- Backbone ---
  shape-matched backbone keys: 299 / 299
  first 5 matched keys: ['model.embed_tokens.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.kv_a_proj_with_mqa.weight', 'model.layers.0.self_attn.kv_a_layernorm.weight', 'model.layers.0.self_attn.kv_b_proj.weight', 'model.layers.0.self_attn.o_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.input_layernorm.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.self_attn.q_proj.weight', 'model.layers.1.self_attn.kv_a_proj_with_mqa.weight', 'model.layers.1.self_attn.kv_a_layernorm.weight', 'model.layers.1.self_attn.kv_b_proj.weight', 'model.layers.1.self_attn.o_proj.weight', 'model.layers.1.mlp.gate.weight', 'model.layers.1.mlp.shared_experts.gate_proj.weight', 'model.layers.1.mlp.shared_experts.up_proj.weight', 'model.layers.1.mlp.shared_experts.down_proj.weight', 'model.layers.1.inp

## [5/5] Forward pass smoke test (CPU)

Tokenize a short prompt, run a single forward pass through the partial model with `torch.no_grad()`, and verify the logits tensor has shape `(batch, seq, vocab)` and contains only finite values. Most tokens will route to experts the partial model doesn't own — `myceliaSparseMoeBlock` silently skips those contributions, so the logits aren't semantically meaningful. This is a structural check that the partial model is runnable end-to-end.

In [35]:
print("\n[5/5] Forward pass smoke test (CPU)")
_run_forward_pass(partial_model, config)

print("\n" + "=" * 72)
print("ALL CHECKS PASSED")
print("=" * 72)


[5/5] Forward pass smoke test (CPU)


  prompt        = 'Hello, world!'
  input_ids     = shape (1, 4), dtype torch.int64
  logits        = shape (1, 4, 102400), dtype torch.bfloat16
  [PASS] logits shape is (batch, seq, vocab) = (1, 4, 102400)
  [PASS] all logits are finite (no NaN / Inf)

ALL CHECKS PASSED
